# Chapter 17: Differential Geometry Of Curves

Source orientation: printed pages 307-327; PDF pages 325-345. I inspected the rendered source span privately for structure and terminology: vector products in Euclidean space, vector functions and derivatives, curvature, evolutes and involutes, the equiangular spiral, the catenary, the tractrix, twisted curves, Serret-Frenet formulas, circular and general helices, and concho-spirals. This notebook is original teaching material and does not reproduce textbook prose, exercises, screenshots, page crops, or source figures.

## Chapter Goal

Differential geometry turns a curve from a drawn trace into a moving coordinate system. A point has a position vector. Its derivative gives velocity; after reparameterizing by arc length, that velocity becomes the unit tangent. The tangent's derivative records curvature. In space, a curve needs a whole moving trihedron: tangent, principal normal, and binormal. Their derivatives encode curvature and torsion, the two rates that describe how a curve bends and twists.

The notebook follows the chapter's progression. First, vector products are made geometric through projections, areas, and signed volumes. Then vector derivatives are tested on moving vectors of constant length. Plane curvature is computed for an ellipse, and the evolute is drawn as the locus of centers of curvature. The catenary and tractrix are paired because the source treats the tractrix as an involute of the catenary. The circular helix becomes a laboratory for the Serret-Frenet equations, with constant curvature and torsion checked numerically. Finally, general helices and concho-spirals show how fixed-angle motion on a cylinder or cone extends the helix idea.

The core inspection target is local: at a chosen point on a curve, which vector points along the curve, which vector points toward bending, and which vector records twisting away from the osculating plane?

## Visualization Storyboard

| Section | Representation | Artifact | Inspection target | Check |
|---|---|---|---|---|
| Vector products | projection, cross-product area, triple-product volume | `vector-products-projection-volume.png` | dot, cross, and scalar triple products have geometric meanings | Gram/determinant identities |
| Vector derivatives | moving unit vector on circle | `vector-function-derivative-orthogonality.png` | a constant-length vector has derivative perpendicular to itself | `r dot r' = 0` |
| Curvature and evolute | ellipse with normals and evolute | `ellipse-curvature-evolute.png` | centers of curvature form the evolute | numeric formula versus known evolute |
| Involutes | catenary and tractrix | `catenary-tractrix-involute.png` | tractrix tangent segment length is constant | tangent-length residual |
| Twisted curves | 3D circular helix with Frenet frames | `circular-helix-frenet-frame.html` | curvature and torsion are constant | finite-difference and exact formulas |
| General helices and concho-spiral | tangent-angle and conical spiral view | `general-helix-concho-spiral.png` | fixed tangent angle and inverse curvature/torsion pattern | ratio and cone-generator checks |

Matplotlib is used for exact 2D annotations and comparison panels. Plotly is used for the spatial helix so the moving trihedron can be rotated. NumPy provides the vector calculus and finite-difference checks; the formulas are kept visible in notebook cells rather than hidden in a generator.

In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import sys

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Polygon
import numpy as np
import plotly.graph_objects as go
from IPython.display import Markdown, display

CHAPTER_NO = 17
HERE = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [HERE, *HERE.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not locate the Introduction-to-Geometry book root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, display_artifact, ensure_chapter_artifact_dirs, write_csv, write_json
from utils.course import chapter_by_no

CHAPTER = chapter_by_no(CHAPTER_NO)
DIRS = ensure_chapter_artifact_dirs(CHAPTER_NO, BOOK_ROOT)
FIG = DIRS["figures"]
HTML = DIRS["html"]
CHECKS = DIRS["checks"]
TABLES = DIRS["tables"]

for stale in [FIG / "concept_configuration.svg", FIG / "parameter_experiment.svg", TABLES / "artifact_manifest.csv"]:
    if stale.exists():
        stale.unlink()

COLORS = {
    "ink": "#111827",
    "muted": "#6b7280",
    "blue": "#2563eb",
    "green": "#059669",
    "orange": "#d97706",
    "red": "#dc2626",
    "purple": "#7c3aed",
}

visual_artifacts: list[Path] = []
check_artifacts: list[Path] = []
table_artifacts: list[Path] = []
computed_checks: dict[str, object] = {}

def save_png(fig: plt.Figure, filename: str) -> Path:
    path = FIG / filename
    fig.savefig(path, dpi=165, bbox_inches="tight")
    plt.close(fig)
    visual_artifacts.append(path)
    return path

def display_many(paths: list[Path], width: int = 760) -> None:
    for path in paths:
        display(Markdown(f"### {path.name}"))
display_artifact(path, width=width)

def rel(path: Path) -> str:
    return path.resolve().relative_to(BOOK_ROOT.resolve()).as_posix()

def unit(v: np.ndarray) -> np.ndarray:
    return v / np.linalg.norm(v, axis=-1, keepdims=True)

def arrow2(ax, start, vec, color, label=None, scale=1.0):
    end = np.array(start) + scale * np.array(vec)
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle="->", mutation_scale=13, lw=1.8, color=color))
    if label:
        ax.text(end[0] + 0.04, end[1] + 0.04, label, color=color, fontsize=10)


## 1. Vector Products: Projection, Area, Volume

The chapter begins with vector algebra because differential geometry constantly asks how a moving direction is changing. The dot product measures projection and orthogonality. The cross product measures oriented area and supplies a normal direction. The scalar triple product measures signed volume and tests coplanarity.

The figure below uses one asymmetric configuration so the signs and magnitudes cannot hide in symmetry. The accompanying checks verify two important identities: the squared area of a parallelogram is the Gram determinant `|a|^2 |b|^2 - (a dot b)^2`, and the squared volume of a parallelepiped is the determinant of the Gram matrix of the three edge vectors.

In [ ]:
a = np.array([2.0, 0.25, 0.0])
b = np.array([0.75, 1.65, 0.0])
c = np.array([0.55, 0.35, 1.35])
proj_b_on_a = (np.dot(a, b) / np.dot(a, a)) * a
cross_ab = np.cross(a, b)
triple = float(np.dot(cross_ab, c))
area_cross = float(np.linalg.norm(cross_ab))
area_gram = math.sqrt(float(np.dot(a, a) * np.dot(b, b) - np.dot(a, b) ** 2))
gram = np.array([[np.dot(u, v) for v in [a, b, c]] for u in [a, b, c]])
volume_gram = math.sqrt(float(np.linalg.det(gram)))

fig, axs = plt.subplots(1, 2, figsize=(13, 5.3))
ax = axs[0]
arrow2(ax, [0, 0], a[:2], COLORS["blue"], "a")
arrow2(ax, [0, 0], b[:2], COLORS["green"], "b")
arrow2(ax, [0, 0], proj_b_on_a[:2], COLORS["orange"], "proj_a b")
ax.plot([b[0], proj_b_on_a[0]], [b[1], proj_b_on_a[1]], color=COLORS["muted"], ls="--")
par = np.array([[0, 0], a[:2], (a + b)[:2], b[:2]])
ax.add_patch(Polygon(par, closed=True, facecolor="#bfdbfe", edgecolor=COLORS["blue"], alpha=0.32))
ax.set_title("dot product as projection; cross product as area")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.25)
ax.set_xlabel("x")
ax.set_ylabel("y")

ax = axs[1]
base = np.array([[0, 0], a[:2], (a + b)[:2], b[:2], [0, 0]])
top = base + c[:2]
ax.plot(base[:, 0], base[:, 1], color=COLORS["blue"], lw=2)
ax.plot(top[:, 0], top[:, 1], color=COLORS["purple"], lw=2)
for p, q in zip(base[:-1], top[:-1]):
    ax.plot([p[0], q[0]], [p[1], q[1]], color=COLORS["muted"], lw=1.4)
arrow2(ax, [0, 0], c[:2], COLORS["red"], "c")
ax.text(0.1, 2.8, f"signed volume = {triple:.3f}", fontsize=11)
ax.set_title("scalar triple product as signed volume")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.25)
ax.set_xlabel("projected x")
ax.set_ylabel("projected y")

vector_path = save_png(fig, "vector-products-projection-volume.png")
computed_checks["vector_products"] = {
    "dot_ab": float(np.dot(a, b)),
    "cross_area": area_cross,
    "gram_area": area_gram,
    "signed_triple_product": triple,
    "gram_volume": volume_gram,
}
assert abs(area_cross - area_gram) < 1e-12
assert abs(abs(triple) - volume_gram) < 1e-12
display_many([vector_path])


## 2. Vector Functions And Derivatives

A vector function `r(s)` can be differentiated component by component, but the geometry of the derivative is more important than the syntax. If a vector keeps constant length, then differentiating `r dot r = constant` gives `r dot r' = 0`. The derivative must be perpendicular to the vector.

The moving vector on the unit circle is the cleanest laboratory. The position vector points radially outward; the derivative points tangentially. The acceleration points back inward. This is the same pattern the chapter later uses to define tangents, normals, and curvature for arbitrary curves.

In [ ]:
t = np.linspace(0, 2 * np.pi, 400)
circle = np.column_stack([np.cos(t), np.sin(t)])
selected = np.array([0.25, 1.15, 2.35, 3.7, 5.05])

fig, ax = plt.subplots(figsize=(7.2, 7.0))
ax.plot(circle[:, 0], circle[:, 1], color=COLORS["blue"], lw=2.4)
orth_rows = []
for s in selected:
    r = np.array([math.cos(s), math.sin(s)])
    rp = np.array([-math.sin(s), math.cos(s)])
    rpp = -r
    arrow2(ax, [0, 0], r, COLORS["green"], "r" if s == selected[0] else None, scale=0.9)
    arrow2(ax, r, rp, COLORS["orange"], "r'" if s == selected[0] else None, scale=0.38)
    arrow2(ax, r, rpp, COLORS["red"], "r''" if s == selected[0] else None, scale=0.28)
    orth_rows.append({"s": float(s), "dot_r_rprime": float(np.dot(r, rp)), "speed": float(np.linalg.norm(rp)), "acceleration_radial_dot": float(np.dot(rpp, r))})
ax.set_aspect("equal", adjustable="box")
ax.set_title("constant length forces derivative perpendicular to the vector")
ax.grid(alpha=0.25)
ax.set_xlabel("x")
ax.set_ylabel("y")

derivative_path = save_png(fig, "vector-function-derivative-orthogonality.png")
derivative_table = write_csv(TABLES / "unit-vector-derivative-checks.csv", orth_rows)
table_artifacts.append(derivative_table)
computed_checks["vector_derivatives"] = {
    "max_abs_dot_r_rprime": max(abs(row["dot_r_rprime"]) for row in orth_rows),
    "speed_values": [row["speed"] for row in orth_rows],
}
assert computed_checks["vector_derivatives"]["max_abs_dot_r_rprime"] < 1e-12
display_many([derivative_path])


## 3. Curvature And Evolutes

For a plane curve parameterized by arc length, curvature is the rate at which the unit tangent turns. A circle of radius `rho` has curvature `1/rho`; for a general curve the circle of curvature changes from point to point. The center of that circle traces the evolute.

The ellipse is a useful test because its evolute has a known astroid-like formula. The code computes curvature and centers of curvature from derivatives, overlays normals, and compares the numerical evolute with the closed form. This turns the source formula `r_c = r + rho n` into a visible object.

In [ ]:
A, B = 2.2, 1.15
u = np.linspace(0.05, 2 * np.pi - 0.05, 700)
x = A * np.cos(u)
y = B * np.sin(u)
xp = -A * np.sin(u)
yp = B * np.cos(u)
xpp = -A * np.cos(u)
ypp = -B * np.sin(u)
speed2 = xp**2 + yp**2
cross = xp * ypp - yp * xpp
curvature = cross / (speed2 ** 1.5)
evolute_x = x - yp * speed2 / cross
evolute_y = y + xp * speed2 / cross
known_x = (A * A - B * B) / A * np.cos(u) ** 3
known_y = (B * B - A * A) / B * np.sin(u) ** 3
evolute_error = np.max(np.hypot(evolute_x - known_x, evolute_y - known_y))

fig, axs = plt.subplots(1, 2, figsize=(13, 5.5))
ax = axs[0]
ax.plot(x, y, color=COLORS["blue"], lw=2.4, label="ellipse")
ax.plot(evolute_x, evolute_y, color=COLORS["red"], lw=2.0, label="evolute")
for idx in np.linspace(40, len(u) - 60, 9, dtype=int):
    p = np.array([x[idx], y[idx]])
    pc = np.array([evolute_x[idx], evolute_y[idx]])
    ax.plot([p[0], pc[0]], [p[1], pc[1]], color=COLORS["muted"], lw=1.0, alpha=0.65)
ax.set_aspect("equal", adjustable="box")
ax.set_title("centers of curvature trace the evolute")
ax.grid(alpha=0.25)
ax.legend()
ax.set_xlabel("x")
ax.set_ylabel("y")

ax = axs[1]
ax.plot(u, curvature, color=COLORS["purple"], lw=2.2)
ax.axhline(0, color=COLORS["ink"], lw=0.8)
ax.set_title("signed curvature varies along the ellipse")
ax.set_xlabel("parameter u")
ax.set_ylabel("curvature")
ax.grid(alpha=0.25)

evolute_path = save_png(fig, "ellipse-curvature-evolute.png")
curvature_rows = []
for idx in np.linspace(60, len(u) - 80, 8, dtype=int):
    curvature_rows.append({"u": float(u[idx]), "curvature": float(curvature[idx]), "radius_of_curvature": float(1 / curvature[idx]), "evolute_x": float(evolute_x[idx]), "evolute_y": float(evolute_y[idx])})
curvature_table = write_csv(TABLES / "ellipse-curvature-samples.csv", curvature_rows)
table_artifacts.append(curvature_table)
computed_checks["curvature_evolute"] = {
    "max_evolute_formula_error": float(evolute_error),
    "min_curvature": float(curvature.min()),
    "max_curvature": float(curvature.max()),
}
assert evolute_error < 1e-10
display_many([evolute_path])


## 4. Catenary And Tractrix As Evolute/Involute Companions

The catenary is derived from the equilibrium of a hanging chain, and the tractrix appears as an involute obtained by unwinding from the catenary. With parameter `u` and scale `a`, the catenary is `(a u, a cosh u)`, while the tractrix is `(a(u - tanh u), a sech u)`.

The defining tractrix property is concrete: the tangent segment from the point of contact to the `x`-axis has constant length `a`. The figure shows the catenary, the tractrix, and several tangent segments. The check table computes the tangent length from the formula rather than trusting the drawing.

In [ ]:
a_scale = 1.0
u = np.linspace(-2.4, 2.4, 500)
catenary = np.column_stack([a_scale * u, a_scale * np.cosh(u)])
sech = 1 / np.cosh(u)
tractrix = np.column_stack([a_scale * (u - np.tanh(u)), a_scale * sech])

fig, ax = plt.subplots(figsize=(10, 6.2))
ax.plot(catenary[:, 0], catenary[:, 1], color=COLORS["blue"], lw=2.4, label="catenary")
ax.plot(tractrix[:, 0], tractrix[:, 1], color=COLORS["red"], lw=2.4, label="tractrix")
tangent_rows = []
for val in [-2.0, -1.2, -0.45, 0.45, 1.2, 2.0]:
    q = np.array([a_scale * (val - math.tanh(val)), a_scale / math.cosh(val)])
    dx = math.tanh(val) ** 2
    dy = -(1 / math.cosh(val)) * math.tanh(val)
    tangent = np.array([dx, dy]) / math.hypot(dx, dy)
    signed_length_to_axis = -q[1] / tangent[1]
    foot = q + signed_length_to_axis * tangent
    ax.plot([q[0], foot[0]], [q[1], foot[1]], color=COLORS["muted"], lw=1.3)
    ax.scatter([q[0], foot[0]], [q[1], foot[1]], color=[COLORS["red"], COLORS["ink"]], s=28)
    tangent_rows.append({"u": val, "tangent_length_to_x_axis": float(abs(signed_length_to_axis)), "foot_x": float(foot[0])})
ax.axhline(0, color=COLORS["ink"], lw=1.0)
ax.set_title("tractrix tangent segments have constant length")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.25)
ax.legend()

tractrix_path = save_png(fig, "catenary-tractrix-involute.png")
tractrix_table = write_csv(TABLES / "tractrix-tangent-lengths.csv", tangent_rows)
table_artifacts.append(tractrix_table)
computed_checks["catenary_tractrix"] = {
    "target_length": a_scale,
    "max_tangent_length_error": max(abs(row["tangent_length_to_x_axis"] - a_scale) for row in tangent_rows),
}
assert computed_checks["catenary_tractrix"]["max_tangent_length_error"] < 1e-12
display_many([tractrix_path])


## 5. Twisted Curves And The Circular Helix

A space curve needs more than a tangent. The Serret-Frenet frame consists of the unit tangent `t`, principal normal `p`, and binormal `b = t x p`. Curvature measures how quickly `t` turns toward `p`; torsion measures how quickly the osculating plane twists.

The circular helix is the ideal laboratory because both curvature and torsion are constant. For `r(u) = (a cos u, a sin u, c u)`, arc length grows at rate `sqrt(a^2 + c^2)`, and the exact formulas are `kappa = a/(a^2+c^2)` and `tau = c/(a^2+c^2)`. The Plotly artifact places several Frenet frames along the helix.

In [ ]:
radius, pitch = 1.0, 0.42
u = np.linspace(0, 6 * np.pi, 700)
helix = np.column_stack([radius * np.cos(u), radius * np.sin(u), pitch * u])
speed = math.sqrt(radius * radius + pitch * pitch)
tangent = np.column_stack([-radius * np.sin(u), radius * np.cos(u), np.full_like(u, pitch)]) / speed
principal = np.column_stack([-np.cos(u), -np.sin(u), np.zeros_like(u)])
binormal = np.cross(tangent, principal)
kappa_exact = radius / (radius * radius + pitch * pitch)
tau_exact = pitch / (radius * radius + pitch * pitch)

fig3 = go.Figure()
fig3.add_trace(go.Scatter3d(x=helix[:, 0], y=helix[:, 1], z=helix[:, 2], mode="lines", line=dict(color="#2563eb", width=5), name="circular helix"))
theta = np.linspace(0, 2 * np.pi, 60)
for z0 in np.linspace(helix[:, 2].min(), helix[:, 2].max(), 8):
    fig3.add_trace(go.Scatter3d(x=radius * np.cos(theta), y=radius * np.sin(theta), z=np.full_like(theta, z0), mode="lines", line=dict(color="#d1d5db", width=1), showlegend=False))
for idx in np.linspace(60, len(u) - 80, 7, dtype=int):
    base = helix[idx]
    for vec, color, name in [(tangent[idx], "#059669", "t"), (principal[idx], "#dc2626", "p"), (binormal[idx], "#7c3aed", "b")]:
        end = base + 0.45 * vec
        fig3.add_trace(go.Scatter3d(x=[base[0], end[0]], y=[base[1], end[1]], z=[base[2], end[2]], mode="lines", line=dict(color=color, width=5), name=name, showlegend=False))
fig3.update_layout(title="Circular helix with Serret-Frenet frames", width=850, height=650, template="plotly_white", scene=dict(aspectmode="data", xaxis_title="x", yaxis_title="y", zaxis_title="z"))
helix_path = HTML / "circular-helix-frenet-frame.html"
if helix_path.exists():
    helix_path.unlink()
fig3.write_html(helix_path, include_plotlyjs="cdn", full_html=True)
visual_artifacts.append(helix_path)

du = u[1] - u[0]
dt_du = np.gradient(tangent, du, axis=0)
dp_du = np.gradient(principal, du, axis=0)
kappa_numeric = np.linalg.norm(dt_du, axis=1) / speed
tau_numeric = np.sum(dp_du * binormal, axis=1) / speed
trim = slice(20, -20)
computed_checks["circular_helix"] = {
    "kappa_exact": kappa_exact,
    "tau_exact": tau_exact,
    "kappa_numeric_mean": float(kappa_numeric[trim].mean()),
    "tau_numeric_mean": float(tau_numeric[trim].mean()),
    "max_frame_orthogonality_error": float(np.max(np.abs(np.sum(tangent * principal, axis=1)) + np.abs(np.sum(tangent * binormal, axis=1)) + np.abs(np.sum(principal * binormal, axis=1)))),
}
assert abs(kappa_numeric[trim].mean() - kappa_exact) < 5e-4
assert abs(tau_numeric[trim].mean() - tau_exact) < 5e-4
display_many([helix_path])


## 6. General Helices And Concho-Spirals

A general helix can be characterized in two equivalent ways: the tangent makes a constant angle with a fixed direction, or curvature and torsion have a constant ratio. The circular helix satisfies both with constant curvature and torsion. The concho-spiral is a more flexible relative: in cylindrical coordinates it can be written `r = a mu^u`, `theta = u`, `z = c mu^u`, so it lies on a cone and cuts its generators at a fixed angle. Its curvature and torsion are inversely proportional to distance from the apex.

The final artifact places the circular helix next to a concho-spiral and plots two diagnostics: tangent angle with the vertical axis and the curvature/torsion ratio. This is the notebook's bridge from a single elegant example to a class of curves controlled by fixed-angle motion.

In [ ]:
u = np.linspace(0.05, 5.4 * np.pi, 900)
mu = 1.055
a0, c0 = 0.33, 0.50
r = a0 * mu**u
z = c0 * mu**u
concho = np.column_stack([r * np.cos(u), r * np.sin(u), z])
du = u[1] - u[0]
dr = np.gradient(concho, du, axis=0)
d2r = np.gradient(dr, du, axis=0)
d3r = np.gradient(d2r, du, axis=0)
speed = np.linalg.norm(dr, axis=1)
tvec = dr / speed[:, None]
vertical = np.array([0, 0, 1.0])
angles = np.arccos(np.clip(tvec @ vertical, -1, 1))
cross12 = np.cross(dr, d2r)
kappa = np.linalg.norm(cross12, axis=1) / speed**3
tau = np.einsum("ij,ij->i", cross12, d3r) / np.linalg.norm(cross12, axis=1) ** 2
ratio = kappa / np.abs(tau)

fig = plt.figure(figsize=(13, 8))
ax = fig.add_subplot(2, 2, 1, projection="3d")
ax.plot(radius * np.cos(u), radius * np.sin(u), pitch * u, color=COLORS["blue"], lw=2.2)
ax.set_title("circular helix")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")

ax = fig.add_subplot(2, 2, 2, projection="3d")
ax.plot(concho[:, 0], concho[:, 1], concho[:, 2], color=COLORS["red"], lw=2.2)
for idx in np.linspace(60, len(u) - 120, 6, dtype=int):
    ax.plot([0, concho[idx, 0]], [0, concho[idx, 1]], [0, concho[idx, 2]], color=COLORS["muted"], lw=0.9)
ax.set_title("concho-spiral on a cone")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")

ax = fig.add_subplot(2, 2, 3)
ax.plot(u, np.degrees(angles), color=COLORS["green"], lw=2.0)
ax.set_title("tangent angle with vertical is nearly constant")
ax.set_xlabel("parameter u")
ax.set_ylabel("degrees")
ax.grid(alpha=0.25)

ax = fig.add_subplot(2, 2, 4)
trim = slice(40, -40)
ax.plot(u[trim], ratio[trim], color=COLORS["purple"], lw=2.0)
ax.set_title("curvature/torsion ratio diagnostic")
ax.set_xlabel("parameter u")
ax.set_ylabel("kappa / |tau|")
ax.grid(alpha=0.25)
fig.tight_layout()

general_path = save_png(fig, "general-helix-concho-spiral.png")
general_rows = [
    {"diagnostic": "concho_tangent_angle_std_degrees", "value": float(np.degrees(angles[trim]).std())},
    {"diagnostic": "concho_kappa_tau_ratio_std", "value": float(ratio[trim].std())},
    {"diagnostic": "concho_cone_ratio_z_over_radius_mean", "value": float((z / r).mean())},
    {"diagnostic": "concho_cone_ratio_z_over_radius_std", "value": float((z / r).std())},
]
general_table = write_csv(TABLES / "general-helix-concho-diagnostics.csv", general_rows)
table_artifacts.append(general_table)
computed_checks["general_helix_concho"] = {
    "tangent_angle_std_degrees": float(np.degrees(angles[trim]).std()),
    "kappa_tau_ratio_std": float(ratio[trim].std()),
    "cone_ratio_std": float((z / r).std()),
}
assert computed_checks["general_helix_concho"]["tangent_angle_std_degrees"] < 0.3
assert computed_checks["general_helix_concho"]["cone_ratio_std"] < 1e-12
display_many([general_path])


## Reading The Chapter As Moving Frames

The notebook compresses a long chapter into a single recurring pattern. First choose a curve. Then compute its position vector, tangent, and derivative data. In the plane, curvature determines the circle of curvature and hence the evolute. In space, curvature and torsion determine a moving frame and a Darboux vector. Special curves become special because a diagnostic is constant: the circle has constant curvature, the circular helix has constant curvature and torsion, a general helix has constant curvature/torsion ratio, the tractrix has constant tangent length to an axis, and the concho-spiral keeps a fixed relation to a cone.

The computational experiments are meant to be changed. Alter the ellipse axes, catenary scale, helix pitch, or concho-spiral growth factor, then rerun the checks. A visual relation that survives parameter changes is probably a theorem; one that fails may have depended on a hidden normalization.

In [ ]:
manifest_rows = []
for artifact in visual_artifacts:
    manifest_rows.append({"artifact": rel(artifact), "role": "visual", "concept": artifact.stem.replace("-", " ")})
for artifact in table_artifacts:
    manifest_rows.append({"artifact": rel(artifact), "role": "table", "concept": artifact.stem.replace("-", " ")})
manifest_path = write_csv(TABLES / "artifact-manifest.csv", manifest_rows)
table_artifacts.append(manifest_path)

summary_payload = {
    "chapter": CHAPTER_NO,
    "title": CHAPTER["title"],
    "source_printed_pages": CHAPTER["printed"],
    "source_pdf_pages": CHAPTER["pdf"],
    "visuals": [rel(path) for path in visual_artifacts],
    "tables": [rel(path) for path in table_artifacts],
    "checks": computed_checks,
}
summary_path = write_json(CHECKS / "visual-summary.json", summary_payload)
legacy_summary_path = write_json(CHECKS / "visual_summary.json", summary_payload)
check_artifacts.extend([summary_path, legacy_summary_path])

final_sanity = {
    "visual_count": len(visual_artifacts),
    "table_count": len(table_artifacts),
    "area_identity_error": abs(computed_checks["vector_products"]["cross_area"] - computed_checks["vector_products"]["gram_area"]),
    "evolute_error": computed_checks["curvature_evolute"]["max_evolute_formula_error"],
    "tractrix_length_error": computed_checks["catenary_tractrix"]["max_tangent_length_error"],
    "helix_kappa_error": abs(computed_checks["circular_helix"]["kappa_numeric_mean"] - computed_checks["circular_helix"]["kappa_exact"]),
    "concho_angle_std": computed_checks["general_helix_concho"]["tangent_angle_std_degrees"],
}
final_sanity_path = write_json(CHECKS / "final-sanity-summary.json", final_sanity)
check_artifacts.append(final_sanity_path)

assert_artifacts(visual_artifacts + table_artifacts + check_artifacts, min_bytes=80)
assert final_sanity["visual_count"] >= 6
assert final_sanity["area_identity_error"] < 1e-12
assert final_sanity["evolute_error"] < 1e-10
assert final_sanity["tractrix_length_error"] < 1e-12
assert final_sanity["helix_kappa_error"] < 5e-4
assert final_sanity["concho_angle_std"] < 0.3
assert not (FIG / "concept_configuration.svg").exists()
assert not (FIG / "parameter_experiment.svg").exists()
assert not (TABLES / "artifact_manifest.csv").exists()

summary_payload


## Takeaways

- Dot, cross, and scalar triple products are geometric tools: projection, area, normal direction, and signed volume.
- A constant-length moving vector has derivative perpendicular to itself, which is the local reason circular motion has tangential velocity and radial acceleration.
- Plane curvature measures turning of the unit tangent; centers of curvature trace the evolute.
- The tractrix is an involute of the catenary, and its tangent segment to the axis has constant length.
- A space curve uses the Serret-Frenet frame `t, p, b`; curvature bends the tangent and torsion twists the osculating plane.
- A circular helix has constant curvature and torsion, while a general helix keeps the tangent at a fixed angle to a direction.
- A concho-spiral generalizes helical behavior on a cone, with fixed-angle structure replacing constant radius.